In [39]:
import pandas as pd

Upload the csv downloaded from github here -

In [40]:
df = pd.read_csv('/content/Indian-Male-Names.csv')

In [41]:
df

,name,gender,race
0,barjraj,m,indian
1,ramdin verma,m,indian
2,sharat chandran,m,indian
3,birender mandal,m,indian
4,amit,m,indian
...,...,...,...
14840,buddha,m,indian
14841,mukesh,m,indian
14842,monu,m,indian
14843,govind prasad shahu,m,indian


In [42]:
df["name"] = df["name"].fillna("").str.lower().str.strip()

df["tokens"] = df["name"].str.split()

df = df[df["tokens"].apply(len) == 2]

df["first"] = df["tokens"].apply(lambda x: x[0])
df["last"] = df["tokens"].apply(lambda x: x[-1])


/tmp/ipykernel_1732/843636190.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["first"] = df["tokens"].apply(lambda x: x[0])
/tmp/ipykernel_1732/843636190.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["last"] = df["tokens"].apply(lambda x: x[-1])


In [43]:
df['first']

,first
1,ramdin
2,sharat
3,birender
7,shiv
8,vikram
...,...
14834,jarnail
14836,vinod
14837,raj
14838,sanjya


In [44]:
df = df.sample(5000, random_state=42)

df = df[["first","last"]].reset_index(drop=True)

print(df.head())


     first    last
0   dinesh   kumar
1  sandeep   singh
2  ranjeet   singh
3   vishal  kapoor
4    punit  sharam


In [45]:
print("Dataset size:", len(df))

print("Unique first names:", df['first'].nunique())
print("Unique surnames:", df['last'].nunique())

print(df.head())


Dataset size: 5000
Unique first names: 1791
Unique surnames: 1168
     first    last
0   dinesh   kumar
1  sandeep   singh
2  ranjeet   singh
3   vishal  kapoor
4    punit  sharam


In [46]:
import string

chars = list(string.ascii_lowercase)

special = ["<pad>", "<sos>", "<eos>", "<unk>"]

vocab = chars + special

char2idx = {c:i for i,c in enumerate(vocab)}
idx2char = {i:c for c,i in char2idx.items()}

vocab_size = len(vocab)


In [47]:
MAX_LEN = 12

def encode(name, add_tokens=False):

    tokens = [char2idx.get(c, char2idx["<unk>"]) for c in name]

    if add_tokens:
        tokens = [char2idx["<sos>"]] + tokens + [char2idx["<eos>"]]

    tokens = tokens[:MAX_LEN]

    if len(tokens) < MAX_LEN:
        tokens += [char2idx["<pad>"]] * (MAX_LEN - len(tokens))

    return tokens


In [61]:
def generate_square_subsequent_mask(size):

    mask = torch.tril(torch.ones(size, size))

    return mask


In [48]:
import torch
from torch.utils.data import Dataset

class NameDataset(Dataset):

    def __init__(self, df):
        self.src = df["first"].tolist()
        self.tgt = df["last"].tolist()

    def __len__(self):
        return len(self.src)

    def __getitem__(self, idx):

        src = encode(self.src[idx])
        tgt = encode(self.tgt[idx], add_tokens=True)

        tgt_input = tgt[:-1]
        tgt_output = tgt[1:]

        return (
            torch.tensor(src),
            torch.tensor(tgt_input),
            torch.tensor(tgt_output)
        )


In [49]:
import torch.nn as nn
import math

class PositionalEncoding(nn.Module):

    def __init__(self, d_model, max_len=100):

        super().__init__()

        pe = torch.zeros(max_len, d_model)

        pos = torch.arange(0, max_len).unsqueeze(1)

        div = torch.exp(
            torch.arange(0, d_model, 2) *
            (-math.log(10000.0)/d_model)
        )

        pe[:,0::2] = torch.sin(pos*div)
        pe[:,1::2] = torch.cos(pos*div)

        self.pe = pe.unsqueeze(0)

    def forward(self,x):

        return x + self.pe[:,:x.size(1)]


In [50]:
class MultiHeadAttention(nn.Module):

    def __init__(self,d_model,heads):

        super().__init__()

        self.d_model = d_model
        self.heads = heads
        self.dk = d_model // heads

        self.q = nn.Linear(d_model,d_model)
        self.k = nn.Linear(d_model,d_model)
        self.v = nn.Linear(d_model,d_model)

        self.fc = nn.Linear(d_model,d_model)

    def forward(self, q, k, v, mask=None):
        B, Tq, D = q.shape
        B, Tk, D = k.shape
        B, Tv, D = v.shape

        Q = self.q(q)
        K = self.k(k)
        V = self.v(v)

        Q = Q.view(B, Tq, self.heads, self.dk).transpose(1, 2)
        K = K.view(B, Tk, self.heads, self.dk).transpose(1, 2)
        V = V.view(B, Tv, self.heads, self.dk).transpose(1, 2)

        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.dk)

        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)

        attn = torch.softmax(scores, dim=-1)

        context = attn @ V

        context = context.transpose(1,2).contiguous().view(B, Tq, D)

        return self.fc(context)



In [51]:
class EncoderBlock(nn.Module):

    def __init__(self,d_model,heads,ff_dim):

        super().__init__()

        self.attn = MultiHeadAttention(d_model,heads)

        self.norm1 = nn.LayerNorm(d_model)

        self.ff = nn.Sequential(
            nn.Linear(d_model,ff_dim),
            nn.ReLU(),
            nn.Linear(ff_dim,d_model)
        )

        self.norm2 = nn.LayerNorm(d_model)

    def forward(self,x):

        x = self.norm1(x + self.attn(x,x,x))
        x = self.norm2(x + self.ff(x))

        return x


In [62]:
class DecoderBlock(nn.Module):

    def __init__(self,d_model,heads,ff_dim):

        super().__init__()

        self.self_attn = MultiHeadAttention(d_model,heads)
        self.cross_attn = MultiHeadAttention(d_model,heads)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)

        self.ff = nn.Sequential(
            nn.Linear(d_model,ff_dim),
            nn.ReLU(),
            nn.Linear(ff_dim,d_model)
        )

    def forward(self, x, enc, mask):

        x = self.norm1(x + self.self_attn(x, x, x, mask))

        x = self.norm2(x + self.cross_attn(x, enc, enc))

        x = self.norm3(x + self.ff(x))

        return x


In [63]:
class Transformer(nn.Module):

    def __init__(self,vocab_size,d_model=128,heads=4,layers=2,ff_dim=256):

        super().__init__()

        self.embed = nn.Embedding(vocab_size,d_model)

        self.pos = PositionalEncoding(d_model)

        self.encoder = nn.ModuleList(
            [EncoderBlock(d_model,heads,ff_dim) for _ in range(layers)]
        )

        self.decoder = nn.ModuleList(
            [DecoderBlock(d_model,heads,ff_dim) for _ in range(layers)]
        )

        self.fc = nn.Linear(d_model,vocab_size)

    def forward(self, src, tgt):

        src = self.pos(self.embed(src))
        tgt = self.pos(self.embed(tgt))

        for layer in self.encoder:
            src = layer(src)

        enc = src

        mask = generate_square_subsequent_mask(tgt.size(1))

        for layer in self.decoder:
            tgt = layer(tgt, enc, mask)

        return self.fc(tgt)


In [64]:
dataset = NameDataset(df)

loader = torch.utils.data.DataLoader(dataset,batch_size=64,shuffle=True)

model = Transformer(
    vocab_size,
    d_model=128,
    heads=4,
    layers=3,
    ff_dim=256
)

criterion = nn.CrossEntropyLoss(ignore_index=char2idx["<pad>"])

optimizer = torch.optim.Adam(model.parameters(),lr=1e-3)


Training loop

In [65]:
from tqdm import tqdm

for epoch in range(30):

    model.train()
    total_loss = 0

    progress = tqdm(loader, desc=f"Epoch {epoch+1}")

    for src, tgt_in, tgt_out in progress:

        pred = model(src, tgt_in)

        loss = criterion(
            pred.view(-1, vocab_size),
            tgt_out.view(-1)
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        progress.set_postfix(loss=loss.item())

    avg_loss = total_loss / len(loader)

    print(f"Epoch {epoch+1} | Avg Loss: {avg_loss:.4f}")


Epoch 1: 100%|██████████| 79/79 [00:13<00:00,  5.69it/s, loss=0.779]


Epoch 1 | Avg Loss: 1.6307


Epoch 2: 100%|██████████| 79/79 [00:16<00:00,  4.88it/s, loss=1.68]


Epoch 2 | Avg Loss: 1.2323


Epoch 3: 100%|██████████| 79/79 [00:18<00:00,  4.29it/s, loss=0.885]


...

Epoch 28 | Avg Loss: 0.4189


Epoch 29: 100%|██████████| 79/79 [00:12<00:00,  6.41it/s, loss=0.372]


Epoch 29 | Avg Loss: 0.3979


Epoch 30: 100%|██████████| 79/79 [00:12<00:00,  6.11it/s, loss=0.602]

Epoch 30 | Avg Loss: 0.3853


In [66]:
def generate_surname(model, name, max_len=12):

    model.eval()

    # encode first name
    src = torch.tensor([encode(name)], dtype=torch.long)

    # start token
    tgt = torch.tensor([[char2idx["<sos>"]]], dtype=torch.long)

    with torch.no_grad():

        for _ in range(max_len):

            out = model(src, tgt)

            # last predicted token
            next_token_logits = out[:, -1, :]

            next_token = torch.argmax(next_token_logits, dim=-1)

            tgt = torch.cat(
                [tgt, next_token.unsqueeze(1)],
                dim=1
            )

            if next_token.item() == char2idx["<eos>"]:
                break

    tokens = tgt.squeeze().tolist()

    chars = []

    for t in tokens:
        c = idx2char[t]

        if c in ["<sos>", "<eos>", "<pad>"]:
            continue

        chars.append(c)

    return "".join(chars)


Inference

In [71]:
generate_surname(model, "neel")


'khan'